In [ ]:
# -----------------------------
# A) Create a synthetic pharma PDF
# -----------------------------
def create_pharma_pdf(pdf_path: str) -> None:
    os.makedirs(os.path.dirname(pdf_path), exist_ok=True)
    c = canvas.Canvas(pdf_path, pagesize=A4)
    _, height = A4

    pages = [
        [
            "PHARMA PRODUCT DOSSIER (SYNTHETIC)",
            "",
            "Product: Amoxycillin Capsules 500 mg",
            "Indication: Treatment of bacterial infections (upper respiratory tract, urinary tract).",
            "Mechanism of Action: Beta-lactam antibiotic; inhibits bacterial cell wall synthesis.",
            "Contraindications: Hypersensitivity to penicillins; history of severe allergy.",
            "Warnings: Risk of anaphylaxis; monitor for rash; adjust dose in renal impairment.",
            "",
            "Dosage & Administration:",
            "- Adults: 500 mg every 8 hours.",
            "- Renal impairment: dose adjustment required based on creatinine clearance.",
            "",
            "Adverse Reactions:",
            "- Common: nausea, diarrhea, rash.",
            "- Serious: anaphylaxis, C. difficile-associated diarrhea (rare).",
        ],
        [
            "CLINICAL OVERVIEW (SYNTHETIC)",
            "",
            "Clinical Summary:",
            "Efficacy shown in randomized controlled trials for susceptible organisms.",
            "Non-inferiority demonstrated versus comparator antibiotics in mild-to-moderate infections.",
            "",
            "Drug Interactions:",
            "- Probenecid increases amoxycillin levels by decreasing renal clearance.",
            "- Oral anticoagulants: monitor INR; may increase bleeding risk.",
            "",
            "Storage & Stability:",
            "Store below 25°C. Protect from moisture. Shelf life: 24 months.",
            "",
            "Quality Attributes:",
            "Assay: 95.0% to 105.0% of label claim.",
            "Dissolution: NLT 80% in 45 minutes (Q).",
        ],
        [
            "MANUFACTURING & CMC (SYNTHETIC)",
            "",
            "Manufacturing Process:",
            "Blending -> Encapsulation -> In-process checks -> Packaging.",
            "Critical process parameters: blend uniformity, capsule fill weight, moisture control.",
            "",
            "Specifications:",
            "- Identification: conforms to reference standard.",
            "- Related substances: within limits.",
            "- Microbial limits: complies with pharmacopeial requirements.",
            "",
            "Packaging:",
            "Alu-Alu blister packs; carton labeling includes batch number and expiry date.",
        ],
    ]

    y = height - 60
    for page_lines in pages:
        c.setFont("Helvetica-Bold", 14)
        c.drawString(60, y, page_lines[0])
        y -= 30

        c.setFont("Helvetica", 11)
        for line in page_lines[1:]:
            if y < 80:
                c.showPage()
                y = height - 60
                c.setFont("Helvetica", 11)
            c.drawString(60, y, line)
            y -= 16

        c.showPage()
        y = height - 60

    c.save()


# -----------------------------
# B) Extract text from PDF
# -----------------------------
def extract_pdf_text(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    all_text = []
    for page in doc:
        all_text.append(page.get_text("text"))
    doc.close()

    text = "\n".join(all_text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text


# -----------------------------
# C) Chunk text
# -----------------------------
def chunk_text(text: str, chunk_size: int = 220, overlap: int = 40) -> List[str]:
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i : i + chunk_size]
        chunks.append(" ".join(chunk))
        i += max(1, chunk_size - overlap)
    return chunks


# -----------------------------
# D) Build fine-tuning dataset (anchor=query, positive=chunk)
# -----------------------------
def pick_keywords(chunk: str, k: int = 6) -> List[str]:
    tokens = [t.lower().strip(".,;:()[]%-") for t in chunk.split()]
    tokens = [t for t in tokens if len(t) > 4 and t.isalpha()]
    freq = {}
    for t in tokens:
        freq[t] = freq.get(t, 0) + 1
    return [w for w, _ in sorted(freq.items(), key=lambda x: x[1], reverse=True)[:k]]


def generate_queries_for_chunk(chunk: str) -> List[str]:
    kws = pick_keywords(chunk)
    queries = []
    templates = [
        "What are the key points about {kw}?",
        "Explain {kw} in this document.",
        "What does the document say about {kw}?",
        "Summarize the section related to {kw}.",
        "What are the risks or warnings regarding {kw}?",
    ]
    for kw in kws[:4]:
        for t in templates[:2]:
            queries.append(t.format(kw=kw))

    queries += [
        "What is the recommended dosage and administration?",
        "List contraindications and warnings mentioned.",
        "What are the common adverse reactions?",
        "What are the storage conditions and shelf life?",
        "Describe manufacturing process and key quality attributes.",
    ]

    uniq = []
    for q in queries:
        if q not in uniq:
            uniq.append(q)
    return uniq[:10]


def build_pairs(chunks: List[str], pairs_per_chunk: int = 4, seed: int = 42) -> Dataset:
    random.seed(seed)
    anchors, positives = [], []
    for ch in chunks:
        qs = generate_queries_for_chunk(ch)
        random.shuffle(qs)
        for q in qs[:pairs_per_chunk]:
            anchors.append(q)
            positives.append(ch)
    return Dataset.from_dict({"anchor": anchors, "positive": positives})


# -----------------------------
# E) Save artifacts
# -----------------------------
def save_jsonl(path: str, rows: List[dict]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def main(
    pdf_path: str = "data/pharma_demo.pdf",
    out_dir: str = "data/pharma_ft_data",
):
    os.makedirs(out_dir, exist_ok=True)

    # 1) Create PDF
    create_pharma_pdf(pdf_path)
    print(f"[OK] Created PDF: {pdf_path}")

    # 2) Extract + chunk
    text = extract_pdf_text(pdf_path)
    chunks = chunk_text(text, chunk_size=220, overlap=40)
    print(f"[OK] Extracted chars: {len(text)} | chunks: {len(chunks)}")

    # 3) Build training pairs
    train_ds = build_pairs(chunks, pairs_per_chunk=4, seed=42)
    print(f"[OK] Training pairs: {len(train_ds)}")

    # 4) Save chunks + dataset
    chunks_path = os.path.join(out_dir, "chunks.jsonl")
    save_jsonl(chunks_path, [{"chunk_id": i, "text": ch} for i, ch in enumerate(chunks)])

    ds_path = os.path.join(out_dir, "train_pairs")
    train_ds.save_to_disk(ds_path)

    print(f"[OK] Saved chunks: {chunks_path}")
    print(f"[OK] Saved dataset: {ds_path}")

    # 5) quick peek
    print("\nSample training rows:")
    for i in range(3):
        print({"anchor": train_ds[i]["anchor"], "positive": train_ds[i]["positive"][:120] + "..."})


if __name__ == "__main__":
    main()

In [ ]:
import os
import zipfile

def zip_folder(folder_path: str, zip_path: str):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                # Preserve folder structure inside zip
                arcname = os.path.relpath(file_path, folder_path)
                zipf.write(file_path, arcname)

    print(f"[OK] Zipped folder saved at: {zip_path}")

# ---- Use it ----
folder_to_zip = "/content/data"
output_zip = "pharma_ft_data_backup.zip"

zip_folder(folder_to_zip, output_zip)